# Chaîne ETL du datamart BTK

Consolidation des données opérationnelles en un **datamart en étoile agrégé par
agence**, qui alimente le tableau de bord, Power BI et la segmentation.

**Exécution :** menu *Kernel → Restart & Run All*, ou `Maj + Entrée` cellule par cellule.

La source est choisie automatiquement, dans cet ordre :
1. **Oracle** — si les variables `BTK_DB_USER`, `BTK_DB_PWD`, `BTK_DB_DSN` sont définies ;
2. **CSV** — si les cinq exports de `etl/export_sources.sql` sont dans `etl/source/` ;
3. **extrait agrégé réel** — le relevé des 49 entités du réseau livré avec le projet.

## 0. Préparation

In [1]:
import os, sys
import numpy as np
import pandas as pd

# Racine du projet : on remonte depuis le dossier courant jusqu'au dossier qui
# contient à la fois "etl" et "clustering".
RACINE = os.path.abspath(os.getcwd())
while not all(os.path.isdir(os.path.join(RACINE, d)) for d in ("etl", "clustering")):
    parent = os.path.dirname(RACINE)
    if parent == RACINE:
        raise SystemExit("Racine du projet introuvable : ouvrez le notebook "
                         "depuis le dossier du projet.")
    RACINE = parent
sys.path.insert(0, os.path.join(RACINE, "etl"))
pd.set_option("display.width", 160, "display.max_columns", 20)
print("Racine du projet :", RACINE)
print("pandas", pd.__version__, "| numpy", np.__version__)

Racine du projet : /home/user/oussema_korbosli
pandas 3.0.5 | numpy 2.4.6


## 1. Collecte (*Extract*)

Lecture des cinq tables sources `AGENCE`, `B_UTILISATEURS`, `CLIENT_BTK`,
`B_OBJECTIF` et `POINTAGE`. La sélection de la source est déléguée au module
`etl_agences`, pour que notebook et script lisent exactement la même chose.

> **Pour lire directement votre base Oracle**, exécuter d'abord `pip install oracledb`
> puis, dans une cellule placée **avant** celle-ci :
> ```python
> import os
> os.environ["BTK_DB_USER"] = "votre_user"
> os.environ["BTK_DB_PWD"]  = "votre_mot_de_passe"
> os.environ["BTK_DB_DSN"]  = "localhost:1521/FREEPDB1"
> ```
> La cellule suivante basculera d'elle-même sur Oracle, et le datamart gagnera
> le taux de présence, le district et l'axe gestionnaire.

In [2]:
import etl_agences as etl

mode, tables = etl.extract("auto")
if mode == "brut":
    print(" | ".join(f"{n} : {len(tables[n])}" for n in etl.TABLES))
    affichage = tables["agences"].head(10)
else:
    print(f"{len(tables['extrait'])} entités du réseau")
    affichage = tables["extrait"].head(10)
affichage

[extract] source : extrait agrégé réel (clustering/data/agences_reelles.csv)
49 entités du réseau


,SK_AGENCE,agence,nb_gestionnaires,effectif,nb_clients,total_comptes,production_credits,collecte_epargne
0,1,BIZERTE,8,10,1795,799,6300000,812500
1,2,MGHIRA,6,9,1791,607,5880000,690000
2,3,MONASTIR,10,12,1497,557,7000000,450000
3,4,GROMBALIA,6,12,1451,618,5400000,662500
4,5,BEN AROUS,7,18,1313,708,8300000,720000
5,6,CENTRALE,9,26,1226,528,59400000,420000
6,7,NABEUL,7,8,1225,669,6350000,840000
7,8,SFAX 2,6,15,1154,436,4750000,385000
8,9,LA MARSA,5,8,1126,618,6250000,662500
9,10,PALMARIUM,5,13,1118,712,8100000,732500


## 2. Nettoyage

Les enregistrements sans agence de rattachement (`SK_AGENCE` manquant) sont
écartés, les colonnes sont typées et les mesures converties en numérique.

*(Étape sans objet lorsque la source est l'extrait déjà agrégé : il ne contient
qu'une ligne par agence, sans clé manquante.)*

In [3]:
if mode == "brut":
    tables = etl.nettoyer(tables)
    print("employés retenus :", len(tables["employes"]),
          "| clients :", len(tables["clients"]),
          "| lignes d'objectifs :", len(tables["objectifs"]))
else:
    print("Source déjà agrégée par agence : aucune ligne à écarter.")

Source déjà agrégée par agence : aucune ligne à écarter.


## 3 et 4. Transformation et intégration

Agrégation par agence puis fusion sur la clé `SK_AGENCE` :

| Mesure | Calcul |
|---|---|
| `effectif` | nombre d'employés de l'agence |
| `nb_gestionnaires` | employés avec `EST_GESTIONNAIRE = 1` |
| `nb_clients` | nombre de clients rattachés |
| `total_comptes` | somme des ouvertures de comptes |
| `production_credits` | somme de la production de crédits |
| `collecte_epargne` | somme de l'épargne additionnelle |
| `taux_presence` | part des pointages « présent » ou « retard » |

In [4]:
datamart = etl.transformer(mode, tables)
mesures = etl.MESURES + ([etl.PRESENCE] if etl.PRESENCE in datamart.columns else [])
print(f"datamart : {len(datamart)} agences x {len(mesures)} mesures")
datamart.sort_values("nb_clients", ascending=False).head(10)[["agence"] + mesures]

datamart : 49 agences x 6 mesures


,agence,effectif,nb_gestionnaires,nb_clients,total_comptes,production_credits,collecte_epargne
0,BIZERTE,10,8,1795,799,6300000,812500
1,MGHIRA,9,6,1791,607,5880000,690000
2,MONASTIR,12,10,1497,557,7000000,450000
3,GROMBALIA,12,6,1451,618,5400000,662500
4,BEN AROUS,18,7,1313,708,8300000,720000
5,CENTRALE,26,9,1226,528,59400000,420000
6,NABEUL,8,7,1225,669,6350000,840000
7,SFAX 2,15,6,1154,436,4750000,385000
8,LA MARSA,8,5,1126,618,6250000,662500
9,PALMARIUM,13,5,1118,712,8100000,732500


## 5. Contrôle de qualité

Avant chargement : aucune valeur manquante, aucune valeur négative, aucune
agence en double. Le chargement est interrompu si un contrôle échoue.

In [5]:
print("valeurs manquantes :", int(datamart[mesures].isna().sum().sum()))
print("valeurs négatives  :", int((datamart[mesures] < 0).sum().sum()))
print("agences en double  :", int(datamart["agence"].duplicated().sum()))
datamart[mesures].describe().round(1)

valeurs manquantes : 0
valeurs négatives  : 0
agences en double  : 0


,effectif,nb_gestionnaires,nb_clients,total_comptes,production_credits,collecte_epargne
count,49.0,49.0,49.0,49.0,49.0,49.0
mean,20.3,7.1,611.1,428.1,7718138.3,478939.3
std,76.0,10.4,566.3,252.4,10332885.3,517198.8
min,0.0,0.0,0.0,0.0,0.0,0.0
25%,5.0,4.0,2.0,333.0,3800000.0,210000.0
50%,10.0,6.0,724.0,502.0,5324000.0,457500.0
75%,14.0,8.0,1058.0,607.0,6500000.0,650000.0
max,540.0,76.0,1795.0,801.0,59400000.0,3508025.0


## 6. Chargement (*Load*)

Écriture du datamart en étoile dans `etl/entrepot/`, puis du fichier consommé
par la segmentation (`clustering/data/agences.csv`).

In [6]:
etl.load(datamart, mesures, *etl.transformer_gestionnaire(tables))
sorties = sorted(os.listdir(os.path.join(RACINE, "etl", "entrepot")))
print("\nFichiers de l'entrepôt :", ", ".join(sorties))
pd.read_csv(os.path.join(RACINE, "etl", "entrepot", "fait_agence.csv")).head()

[load] dim_agence sans DISTRICT : cet attribut n'est porté que par la table AGENCE (sources Oracle ou CSV).
[load] entrepôt -> etl/entrepot/ (dim_agence, fait_agence)
[load] datamart de segmentation -> clustering/data/agences.csv

Fichiers de l'entrepôt : dim_agence.csv, fait_agence.csv


,SK_AGENCE,effectif,nb_gestionnaires,nb_clients,total_comptes,production_credits,collecte_epargne
0,1,10,8,1795,799,6300000,812500
1,2,9,6,1791,607,5880000,690000
2,3,12,10,1497,557,7000000,450000
3,4,12,6,1451,618,5400000,662500
4,5,18,7,1313,708,8300000,720000


## 7. Vérification

Le notebook doit retrouver **exactement** le résultat du script
`etl/etl_agences.py`. La cellule ci-dessous relance le script et compare.

In [7]:
import subprocess
subprocess.run([sys.executable, os.path.join(RACINE, "etl", "etl_agences.py")],
               cwd=RACINE, capture_output=True, text=True, check=True)
script = pd.read_csv(os.path.join(RACINE, "clustering", "data", "agences.csv"))
identique = script.equals(datamart[["agence"] + mesures].reset_index(drop=True))
print("Notebook et script produisent le même datamart :", identique)
assert identique, "Écart entre le notebook et le script."

Notebook et script produisent le même datamart : True
